# 01 — Collect Open States bills and obtain missing full text

This notebook is the collection half of the project. It searches the six states and nine search terms recovered from the earlier notebooks, keeps every legislative status, enriches each bill with sponsor/action features useful for a later advancement model, and downloads the latest official bill text when the combined abstract contains fewer than 100 words.

It is resumable and deduplicates on the Open States bill ID. It writes only CSV files:

- `candidate_bills_collected.csv` — the master collection and text cache
- `openstates_collection_progress.csv` — page-level resume state

Opening this notebook makes no network requests. Requests begin only when you run the collection cell near the end.


## Security before running

Some older notebooks in this folder contain API keys directly in code cells. Treat those keys as exposed: revoke/rotate them in Open States and Dartmouth Chat, and do not reuse them here.

This cleaned notebook deliberately uses `getpass`, so the Open States key is held only in kernel memory and is not saved in the notebook. A future environment-variable setup is possible, but it is not enabled automatically. If you later choose that approach, set `OPENSTATES_API_KEY` outside Jupyter, read it with `os.environ["OPENSTATES_API_KEY"]`, keep secret-bearing files out of version control, and restart the kernel after changing the variable.


In [1]:
# Run once if these packages are not already installed in the Python 3.13.9 kernel.
%pip install -q pandas requests beautifulsoup4 pypdf


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import date, datetime, timezone
from getpass import getpass
from io import BytesIO
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader


# -----------------------------
# Editable research settings
# -----------------------------
STATES = [
    "Virginia",
    "Illinois",
    "California",
    "Texas",
    "Ohio",
    "Georgia"
]

SEARCH_TERMS = [
    '"data center"',
    "datacenter",
    "hyperscale",
    '"server farm"',
    '"colocation facility"',
    '"large load"',
    "megawatt",
    '"energy generation"',
    "interconnection",
]

# Recovered from the earlier collection notebooks.
START_DATE = date.fromisoformat("2023-01-01")
END_DATE = date.fromisoformat("2026-07-31")

ABSTRACT_MIN_WORDS = 100
PER_PAGE = 20
PAUSE_BETWEEN_REQUESTS = 6.0
RATE_LIMIT_WAITS = [30, 60, 120]

# False resumes unfinished work and skips completed state/term searches.
# Set True when you intentionally want to refresh every completed search.
REFRESH_COMPLETED_QUERIES = False

DATA_DIRECTORY = Path.cwd()
OUTPUT_CSV = DATA_DIRECTORY / "candidate_bills_collected.csv"
PROGRESS_CSV = DATA_DIRECTORY / "openstates_collection_progress.csv"
OPENSTATES_BASE_URL = "https://v3.openstates.org"

print("Master CSV:", OUTPUT_CSV.resolve())
print("Progress CSV:", PROGRESS_CSV.resolve())


Master CSV: /Users/michael/Desktop/QSS20/Final Project/candidate_bills_collected.csv
Progress CSV: /Users/michael/Desktop/QSS20/Final Project/openstates_collection_progress.csv


In [4]:
# Enter a newly rotated key. The value is masked and is not stored in this notebook.
OPENSTATES_API_KEY = getpass.getpass("Open States API key: ").strip()
if not OPENSTATES_API_KEY:
    raise ValueError("An Open States API key is required.")

openstates_session = requests.Session()
openstates_session.headers.update(
    {
        "X-API-KEY": OPENSTATES_API_KEY,
        "User-Agent": "Dartmouth academic data-center-policy research",
    }
)

document_session = requests.Session()
document_session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (compatible; Dartmouth academic "
            "data-center-policy research)"
        )
    }
)
print("Sessions configured. No request has been made yet.")


Open States API key:  ········


Sessions configured. No request has been made yet.


In [5]:
OUTPUT_COLUMNS = [
    "bill_id", "state", "session", "identifier", "title", "abstract",
    "abstract_word_count", "first_action_date", "latest_action_date",
    "latest_passage_date", "openstates_url", "bill_classifications",
    "subjects", "originating_chamber", "sponsor_names",
    "primary_sponsor_names", "sponsor_person_ids", "sponsor_parties",
    "primary_sponsor_parties", "sponsor_count", "primary_sponsor_count",
    "party_count", "action_count", "latest_action_description",
    "latest_action_classifications", "derived_status", "passage_count",
    "vote_event_count", "passage_vote_count", "passed_vote_event_count",
    "failed_vote_event_count", "latest_vote_date", "latest_vote_result",
    "latest_vote_motion", "days_first_to_latest_action", "source_urls",
    "full_text", "full_text_source_url", "full_text_status",
    "full_text_error",
]


def clean_text(value):
    text = str(value or "").replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def word_count(value):
    return len(re.findall(r"\b[\w'-]+\b", clean_text(value)))


def pipe(values):
    return "|".join(sorted({clean_text(v) for v in values if clean_text(v)}))


def parse_date(value):
    try:
        return date.fromisoformat(str(value)[:10])
    except (TypeError, ValueError):
        return None


def json_list(value):
    if isinstance(value, list):
        return value
    return []


def combined_abstract(bill):
    return "\n\n".join(
        dict.fromkeys(
            clean_text(item.get("abstract", ""))
            for item in json_list(bill.get("abstracts"))
            if clean_text(item.get("abstract", ""))
        )
    )


def party_from_sponsorship(sponsorship):
    person = sponsorship.get("person") or {}
    current_role = person.get("current_role") or person.get("currentRole") or {}
    extras = person.get("extras") or {}
    candidates = [
        sponsorship.get("party"), person.get("party"),
        current_role.get("party"), extras.get("party"),
    ]
    memberships = person.get("current_memberships") or person.get("currentMemberships") or []
    for membership in memberships:
        organization = membership.get("organization") or {}
        if str(organization.get("classification", "")).lower() == "party":
            candidates.append(organization.get("name"))
    return next((clean_text(value) for value in candidates if clean_text(value)), "")


def action_features(actions):
    actions = json_list(actions)
    all_classes = []
    passage_count = 0

    for action in actions:
        classes = [
            clean_text(value).lower()
            for value in json_list(action.get("classification"))
        ]
        all_classes.extend(classes)
        passage_count += int("passage" in classes)

    dated = sorted(
        actions,
        key=lambda action: (
            str(action.get("date", "")),
            int(action.get("order", 0) or 0),
        ),
    )
    latest = dated[-1] if dated else {}
    latest_classes = [
        clean_text(value).lower()
        for value in json_list(latest.get("classification"))
    ]
    classes = set(all_classes)

    if classes & {"executive-signature", "became-law", "enacted"}:
        status = "enacted"
    elif "executive-veto" in classes:
        status = "vetoed"
    elif "failure" in classes:
        status = "failed"
    elif "withdrawal" in classes:
        status = "withdrawn"
    elif "passage" in classes:
        status = "passed_chamber_or_legislature"
    elif classes & {"referral", "referral-committee", "committee-passage"}:
        status = "active_in_committee_or_chamber"
    elif classes & {"filing", "introduction", "reading-1"}:
        status = "introduced"
    else:
        status = "unknown_or_active"

    return {
        "action_count": len(actions),
        "latest_action_description": clean_text(latest.get("description", "")),
        "latest_action_classifications": pipe(latest_classes),
        "derived_status": status,
        "passage_count": passage_count,
    }


def vote_features(votes):
    votes = json_list(votes)
    passage_vote_count = 0
    passed_vote_count = 0
    failed_vote_count = 0

    for vote in votes:
        motion_classes = {
            clean_text(value).lower()
            for value in json_list(vote.get("motion_classification"))
        }
        passage_vote_count += int("passage" in motion_classes)
        result = clean_text(vote.get("result", "")).lower()
        passed_vote_count += int(result == "pass")
        failed_vote_count += int(result == "fail")

    dated = sorted(votes, key=lambda vote: str(vote.get("start_date", "")))
    latest = dated[-1] if dated else {}

    return {
        "vote_event_count": len(votes),
        "passage_vote_count": passage_vote_count,
        "passed_vote_event_count": passed_vote_count,
        "failed_vote_event_count": failed_vote_count,
        "latest_vote_date": clean_text(latest.get("start_date", "")),
        "latest_vote_result": clean_text(latest.get("result", "")),
        "latest_vote_motion": clean_text(latest.get("motion_text", "")),
    }


def bill_features(bill, state):
    sponsors = json_list(bill.get("sponsorships"))
    primary = [item for item in sponsors if item.get("primary") is True]
    parties = [party_from_sponsorship(item) for item in sponsors]
    primary_parties = [party_from_sponsorship(item) for item in primary]
    first = parse_date(bill.get("first_action_date", ""))
    latest = parse_date(bill.get("latest_action_date", ""))
    organization = bill.get("from_organization") or bill.get("fromOrganization") or {}
    abstract = combined_abstract(bill)

    row = {
        "bill_id": clean_text(bill.get("id", "")),
        "state": state,
        "session": clean_text(bill.get("session", "")),
        "identifier": clean_text(bill.get("identifier", "")),
        "title": clean_text(bill.get("title", "")),
        "abstract": abstract,
        "abstract_word_count": word_count(abstract),
        "first_action_date": clean_text(bill.get("first_action_date", "")),
        "latest_action_date": clean_text(bill.get("latest_action_date", "")),
        "latest_passage_date": clean_text(bill.get("latest_passage_date", "")),
        "openstates_url": clean_text(bill.get("openstates_url", "")),
        "bill_classifications": pipe(json_list(bill.get("classification"))),
        "subjects": pipe(json_list(bill.get("subject"))),
        "originating_chamber": clean_text(organization.get("classification", "")),
        "sponsor_names": pipe(item.get("name", "") for item in sponsors),
        "primary_sponsor_names": pipe(item.get("name", "") for item in primary),
        "sponsor_person_ids": pipe(
            (item.get("person") or {}).get("id", "") for item in sponsors
        ),
        "sponsor_parties": pipe(parties),
        "primary_sponsor_parties": pipe(primary_parties),
        "sponsor_count": len(sponsors),
        "primary_sponsor_count": len(primary),
        "party_count": len({party for party in parties if party}),
        "days_first_to_latest_action": (
            (latest - first).days if first and latest else ""
        ),
        "source_urls": pipe(
            item.get("url", "") for item in json_list(bill.get("sources"))
        ),
    }
    row.update(action_features(bill.get("actions")))
    row.update(vote_features(bill.get("votes")))
    return row


In [6]:
def document_candidates(bill):
    candidates = []
    for collection_name in ["versions", "documents"]:
        for item in json_list(bill.get(collection_name)):
            for link in json_list(item.get("links")):
                url = clean_text(link.get("url", ""))
                if url:
                    candidates.append(
                        {
                            "url": url,
                            "media_type": clean_text(link.get("media_type", "")).lower(),
                            "date": clean_text(item.get("date", "")),
                            "collection": collection_name,
                        }
                    )
    return candidates


def choose_latest_document(bill):
    candidates = document_candidates(bill)
    if not candidates:
        return None

    def readable(item):
        descriptor = f"{item['media_type']} {item['url']}".lower()
        return any(token in descriptor for token in ["pdf", "html", "text", "docx", "wordprocessingml"])

    readable_versions = [
        item for item in candidates
        if item["collection"] == "versions" and readable(item)
    ]
    any_versions = [item for item in candidates if item["collection"] == "versions"]
    readable_documents = [item for item in candidates if readable(item)]
    pool = readable_versions or any_versions or readable_documents or candidates
    return max(pool, key=lambda item: item["date"])


def repaired_document_urls(url):
    parts = urlsplit(url)
    if parts.hostname != "beta.ilga.gov":
        return [url]
    query = parse_qs(parts.query)
    doc_name = query.get("DocName", [""])[0]
    document_type = query.get("DocTypeID", [""])[0]
    match = re.match(r"(\d{3})", doc_name)
    ga = match.group(1) if match else ""
    direct = (
        f"https://www.ilga.gov/documents/legislation/{ga}/{document_type}/PDF/{doc_name}.pdf"
        if ga and document_type and doc_name else ""
    )
    current = "https://ilga.gov/Legislation/BillStatus/FullText?" + urlencode(
        {
            "DocTypeID": document_type,
            "DocNum": query.get("DocNum", [""])[0],
            "GAID": query.get("GAID", [""])[0],
            "LegId": query.get("LegID", query.get("LegId", [""]))[0],
            "SessionID": query.get("SessionID", [""])[0],
            "Print": "1",
        }
    )
    return [item for item in [direct, current] if item]


def extract_pdf_text(content):
    reader = PdfReader(BytesIO(content))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = clean_text(page.extract_text() or "")
        if page_text:
            pages.append(f"[PAGE {page_number}]\n{page_text}")
    return "\n\n".join(pages)


def extract_html_text(content):
    soup = BeautifulSoup(content, "html.parser")
    for element in soup(["script", "style", "nav", "footer", "noscript"]):
        element.decompose()
    main = soup.find("main") or soup.find("article") or soup.body or soup
    return clean_text(main.get_text("\n", strip=True))


def extract_docx_text(content):
    with zipfile.ZipFile(BytesIO(content)) as archive:
        xml_content = archive.read("word/document.xml")
    root = ET.fromstring(xml_content)
    paragraphs = []
    for paragraph in root.iter():
        if paragraph.tag.endswith("}p"):
            words = [node.text for node in paragraph.iter() if node.tag.endswith("}t") and node.text]
            if words:
                paragraphs.append("".join(words))
    return clean_text("\n".join(paragraphs))


def download_and_extract(document, maximum_retries=2):
    last_error = None
    for url in repaired_document_urls(document["url"]):
        for attempt in range(1, maximum_retries + 1):
            try:
                response = document_session.get(url, timeout=90)
                response.raise_for_status()
                content_type = response.headers.get("Content-Type", "").lower()
                media_type = document.get("media_type", "").lower()
                url_lower = response.url.lower()
                if response.content[:8].startswith(b"%PDF"):
                    text = extract_pdf_text(response.content)
                elif "wordprocessingml" in content_type or url_lower.endswith(".docx"):
                    text = extract_docx_text(response.content)
                elif "html" in content_type:
                    text = extract_html_text(response.content)
                elif content_type.startswith("text/"):
                    text = clean_text(response.text)
                elif "pdf" in media_type or url_lower.endswith(".pdf"):
                    text = extract_pdf_text(response.content)
                elif "html" in media_type or url_lower.endswith((".html", ".htm")):
                    text = extract_html_text(response.content)
                else:
                    raise ValueError(f"Unsupported document type: {content_type or media_type or 'unknown'}")
                if len(text) < 100:
                    raise ValueError("Downloaded document contained fewer than 100 text characters.")
                return text, response.url
            except Exception as error:
                last_error = error
                if attempt < maximum_retries:
                    time.sleep(2 ** attempt)
    raise RuntimeError(f"Document download or extraction failed: {last_error}")


In [7]:
def api_get(parameters):
    last_error = None
    for request_attempt in range(1, 4):
        response = None
        for rate_attempt, default_wait in enumerate(RATE_LIMIT_WAITS, start=1):
            response = openstates_session.get(
                f"{OPENSTATES_BASE_URL}/bills", params=parameters, timeout=60
            )
            if response.status_code != 429:
                break
            try:
                wait_seconds = int(float(response.headers.get("Retry-After", default_wait)))
            except ValueError:
                wait_seconds = default_wait
            wait_seconds = max(1, min(300, wait_seconds))
            print(f"  Rate limited; waiting {wait_seconds}s ({rate_attempt}/{len(RATE_LIMIT_WAITS)}).")
            time.sleep(wait_seconds)
        else:
            raise RuntimeError("Open States continued returning 429; progress is safe.")

        try:
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            last_error = error
            if response.status_code == 400:
                raise RuntimeError(f"Open States rejected the request: {response.text[:500]}") from error
            if request_attempt < 3:
                time.sleep(2 ** request_attempt)
    raise RuntimeError(f"Open States request failed: {last_error}")


def load_records():
    if not OUTPUT_CSV.exists():
        return {}
    frame = pd.read_csv(OUTPUT_CSV, dtype=str).fillna("")
    if frame["bill_id"].duplicated().any():
        raise ValueError("The existing master CSV contains duplicate bill_id values.")
    return {row["bill_id"]: row for row in frame.to_dict("records")}


def save_records(records):
    frame = pd.DataFrame(records.values()) if records else pd.DataFrame()
    for column in OUTPUT_COLUMNS:
        if column not in frame:
            frame[column] = ""
    frame = frame[OUTPUT_COLUMNS].sort_values(
        ["state", "first_action_date", "identifier"], kind="stable"
    )
    frame.to_csv(OUTPUT_CSV, index=False)


def load_progress():
    columns = ["state", "search_term", "next_page", "completed", "updated_at"]
    if not PROGRESS_CSV.exists():
        return pd.DataFrame(columns=columns)
    frame = pd.read_csv(PROGRESS_CSV, dtype=str).fillna("")
    for column in columns:
        if column not in frame:
            frame[column] = ""
    return frame[columns]


def progress_for(frame, state, search_term):
    mask = (frame["state"] == state) & (frame["search_term"] == search_term)
    if not mask.any():
        frame.loc[len(frame)] = [state, search_term, "1", "False", ""]
        mask = (frame["state"] == state) & (frame["search_term"] == search_term)
    return frame.index[mask][0]


def add_full_text_if_needed(record, bill):
    abstract_words = word_count(record.get("abstract", ""))
    record["abstract_word_count"] = abstract_words
    if abstract_words >= ABSTRACT_MIN_WORDS:
        if not clean_text(record.get("full_text", "")):
            record["full_text_status"] = "not_needed_abstract_at_least_100_words"
            record["full_text_error"] = ""
        return
    if clean_text(record.get("full_text", "")):
        record["full_text_status"] = "downloaded"
        return
    document = choose_latest_document(bill)
    if document is None:
        record["full_text_status"] = "manual_text_needed"
        record["full_text_error"] = "Abstract under 100 words and no document/version link was returned."
        return
    record["full_text_source_url"] = document["url"]
    try:
        text, final_url = download_and_extract(document)
        record.update(
            {
                "full_text": text,
                "full_text_source_url": final_url,
                "full_text_status": "downloaded",
                "full_text_error": "",
            }
        )
    except Exception as error:
        record["full_text_status"] = "manual_text_needed"
        record["full_text_error"] = str(error)


def merge_bill(records, bill, state):
    bill_id = clean_text(bill.get("id", ""))
    if not bill_id:
        return "ignored"
    first = parse_date(bill.get("first_action_date", ""))
    if first is None or not START_DATE <= first <= END_DATE:
        return "outside_window"

    fresh = bill_features(bill, state)
    existing = records.get(bill_id, {})
    preserved = {
        key: existing.get(key, "")
        for key in OUTPUT_COLUMNS
        if key.startswith("full_text") and existing.get(key, "")
    }
    merged = {**existing, **fresh, **preserved}
    add_full_text_if_needed(merged, bill)
    records[bill_id] = merged
    return "new" if not existing else "updated"


def collect_query(records, progress, state, search_term):
    progress_index = progress_for(progress, state, search_term)
    completed = str(progress.at[progress_index, "completed"]).lower() == "true"
    if completed and not REFRESH_COMPLETED_QUERIES:
        print(f"Skipping completed search: {state} | {search_term}")
        return
    page = 1 if REFRESH_COMPLETED_QUERIES else int(progress.at[progress_index, "next_page"] or 1)
    while True:
        print(f"Searching {state} | {search_term} | page {page}")
        parameters = [
            ("jurisdiction", state), ("q", search_term),
            ("action_since", START_DATE.isoformat()), ("sort", "first_action_asc"),
            ("include", "abstracts"), ("include", "versions"),
            ("include", "documents"), ("include", "actions"),
            ("include", "sponsorships"), ("include", "sources"),
            ("include", "votes"),
            ("page", page), ("per_page", PER_PAGE),
        ]
        payload = api_get(parameters)
        results = payload.get("results", [])
        pagination = payload.get("pagination", {})
        counts = {"new": 0, "updated": 0, "outside_window": 0, "ignored": 0}
        reached_end = False
        for bill in results:
            first = parse_date(bill.get("first_action_date", ""))
            reached_end = reached_end or bool(first and first > END_DATE)
            result = merge_bill(records, bill, state)
            counts[result] += 1
        max_page = int(pagination.get("max_page", page))
        done = page >= max_page or not results or reached_end
        progress.at[progress_index, "next_page"] = str(page + 1)
        progress.at[progress_index, "completed"] = str(done)
        progress.at[progress_index, "updated_at"] = datetime.now(timezone.utc).isoformat()
        save_records(records)
        progress.to_csv(PROGRESS_CSV, index=False)
        print(f"  results={len(results)} new={counts['new']} updated={counts['updated']} total={len(records)}")
        if done:
            break
        page += 1
        time.sleep(PAUSE_BETWEEN_REQUESTS)


## Run or resume collection

This is the first cell that calls Open States or downloads official documents. Stop the kernel at any time; each completed API page has already been saved. Bills whose text cannot be retrieved remain in the master CSV with `full_text_status = manual_text_needed` and an explanation in `full_text_error`.


In [8]:
records = load_records()
progress = load_progress()
starting_count = len(records)

try:
    for state in STATES:
        for search_term in SEARCH_TERMS:
            collect_query(records, progress, state, search_term)
            time.sleep(PAUSE_BETWEEN_REQUESTS)
except KeyboardInterrupt:
    print("Stopped by user; page-level progress is already saved.")
except Exception as error:
    print("Stopped with error:", error)
    print("Progress is saved; fix the issue and rerun this cell.")
finally:
    save_records(records)
    progress.to_csv(PROGRESS_CSV, index=False)

print(f"Started with {starting_count:,}; now have {len(records):,} unique bills.")
print("Saved:", OUTPUT_CSV.resolve())


Searching Virginia | "data center" | page 1
  results=20 new=19 updated=0 total=19
Searching Virginia | "data center" | page 2
  results=20 new=19 updated=1 total=38
Searching Virginia | "data center" | page 3
  results=20 new=19 updated=1 total=57
Searching Virginia | "data center" | page 4
  results=20 new=20 updated=0 total=77
Searching Virginia | "data center" | page 5
  results=20 new=20 updated=0 total=97
Searching Virginia | "data center" | page 6
  results=7 new=7 updated=0 total=104
Searching Virginia | datacenter | page 1
  results=0 new=0 updated=0 total=104
Searching Virginia | hyperscale | page 1
  results=0 new=0 updated=0 total=104
Searching Virginia | "server farm" | page 1
  results=1 new=1 updated=0 total=105
Searching Virginia | "colocation facility" | page 1
  results=0 new=0 updated=0 total=105
Searching Virginia | "large load" | page 1
  results=0 new=0 updated=0 total=105
Searching Virginia | megawatt | page 1
  results=20 new=15 updated=2 total=120
Searching Vir

In [10]:
# Quality-control summary (local CSV only; no API calls).
collected = pd.read_csv(OUTPUT_CSV, dtype=str).fillna("")
display(collected["state"].value_counts().rename("bills"))
display(collected["derived_status"].value_counts(dropna=False).rename("bills"))
display(collected["full_text_status"].value_counts(dropna=False).rename("bills"))

manual_text_queue = collected[collected["full_text_status"] == "manual_text_needed"]
print(f"Bills needing manual text troubleshooting: {len(manual_text_queue):,}")
display(
    manual_text_queue[
        ["state", "identifier", "title", "abstract_word_count", "openstates_url", "full_text_error"]
    ].head(25)
)


state
Illinois      310
Virginia      229
California    217
Texas         188
Ohio           85
Georgia        65
Name: bills, dtype: int64

derived_status
active_in_committee_or_chamber    512
enacted                           177
passed_chamber_or_legislature     169
introduced                        143
failed                             46
vetoed                             27
withdrawn                          20
Name: bills, dtype: int64

full_text_status
downloaded                                584
not_needed_abstract_at_least_100_words    384
manual_text_needed                        126
Name: bills, dtype: int64

Bills needing manual text troubleshooting: 126


,state,identifier,title,abstract_word_count,openstates_url,full_text_error
0,California,AB 100,Budget Acts of 2021 and 2022.,67,https://openstates.org/ca/bills/20232024/AB100/,Document download or extraction failed: Downlo...
1,California,AB 103,Budget Acts of 2021 and 2022.,67,https://openstates.org/ca/bills/20232024/AB103/,Document download or extraction failed: Downlo...
2,California,AB 104,Budget Acts of 2022 and 2023.,69,https://openstates.org/ca/bills/20232024/AB104/,Document download or extraction failed: Downlo...
3,California,AB 106,Budget Acts of 2022 and 2023.,70,https://openstates.org/ca/bills/20232024/AB106/,Document download or extraction failed: Downlo...
9,California,AB 158,Budget Acts of 2022 and 2023.,69,https://openstates.org/ca/bills/20232024/AB158/,Document download or extraction failed: Downlo...
13,California,SB 103,Budget Acts of 2021 and 2022.,67,https://openstates.org/ca/bills/20232024/SB103/,Document download or extraction failed: Downlo...
14,California,SB 104,Budget Acts of 2022 and 2023.,69,https://openstates.org/ca/bills/20232024/SB104/,Document download or extraction failed: Downlo...
15,California,SB 106,Budget Acts of 2022 and 2023.,70,https://openstates.org/ca/bills/20232024/SB106/,Document download or extraction failed: Downlo...
22,California,SB 158,Budget Acts of 2022 and 2023.,69,https://openstates.org/ca/bills/20232024/SB158/,Document download or extraction failed: Downlo...
24,California,ACR 6,Relative to National School Counseling Week.,16,https://openstates.org/ca/bills/20232024/ACR6/,Document download or extraction failed: Downlo...
